In [1]:
# ============================================================
# 06_visualize_image_mask_pairs.ipynb
# ============================================================

!pip install nibabel pandas matplotlib -q

import os
import random
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
BASE = "/content/drive/MyDrive/MELA"

LOG_PATH = os.path.join(BASE, "annotations", "mela_pseudomask_log.csv")

TRAIN_IMG_DIR = os.path.join(BASE, "images", "train")
VAL_IMG_DIR   = os.path.join(BASE, "images", "val")

TRAIN_MASK_DIR = os.path.join(BASE, "masks", "train")
VAL_MASK_DIR   = os.path.join(BASE, "masks", "val")

print("LOG_PATH:", LOG_PATH)
print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR)
print("VAL_IMG_DIR:", VAL_IMG_DIR)
print("TRAIN_MASK_DIR:", TRAIN_MASK_DIR)
print("VAL_MASK_DIR:", VAL_MASK_DIR)

# ------------------------------------------------------------
# LOAD LOG
# ------------------------------------------------------------
df_log = pd.read_csv(LOG_PATH)
df_log.columns = [c.strip() for c in df_log.columns]

print("\nLog shape:", df_log.shape)
print("\nStatus counts:")
print(df_log["status"].value_counts(dropna=False))

print("\nMethod counts:")
print(df_log["method_used"].value_counts(dropna=False))

display(df_log.head())

# ------------------------------------------------------------
# REBUILD PATHS FROM public_id + split
# (mask_path_drive kolonuna bağımlı kalmamak için)
# ------------------------------------------------------------
def build_image_path(public_id, split):
    if split == "train":
        return os.path.join(TRAIN_IMG_DIR, public_id + ".nii.gz")
    elif split == "val":
        return os.path.join(VAL_IMG_DIR, public_id + ".nii.gz")
    else:
        return None

def build_mask_path(public_id, split):
    if split == "train":
        return os.path.join(TRAIN_MASK_DIR, public_id + "_mask.nii.gz")
    elif split == "val":
        return os.path.join(VAL_MASK_DIR, public_id + "_mask.nii.gz")
    else:
        return None

df_log["image_path_check"] = df_log.apply(
    lambda r: build_image_path(r["public_id"], r["split"]),
    axis=1
)

df_log["mask_path_check"] = df_log.apply(
    lambda r: build_mask_path(r["public_id"], r["split"]),
    axis=1
)

df_log["image_exists"] = df_log["image_path_check"].apply(os.path.exists)
df_log["mask_exists"]  = df_log["mask_path_check"].apply(os.path.exists)

print("\nImage exists counts:")
print(df_log["image_exists"].value_counts(dropna=False))

print("\nMask exists counts:")
print(df_log["mask_exists"].value_counts(dropna=False))

# Sadece gerçekten var olan ve status=ok olanları kullan
df_vis = df_log[
    (df_log["status"] == "ok") &
    (df_log["image_exists"] == True) &
    (df_log["mask_exists"] == True)
].copy()

print("\nVisualization-ready sample count:", len(df_vis))
print(df_vis["method_used"].value_counts(dropna=False))

# ------------------------------------------------------------
# HELPER: LOAD NIFTI
# ------------------------------------------------------------
def load_nifti(path):
    nii = nib.load(path)
    data = nii.get_fdata()
    return nii, data

# ------------------------------------------------------------
# HELPER: PICK DISPLAY SLICE
# mask doluysa orta dolu slice, değilse orta slice
# ------------------------------------------------------------
def choose_display_slice(mask_data):
    mask_slices = np.where(mask_data.sum(axis=(0, 1)) > 0)[0]
    if len(mask_slices) > 0:
        return int(mask_slices[len(mask_slices) // 2])
    return mask_data.shape[2] // 2

# ------------------------------------------------------------
# HELPER: VISUALIZE ONE SAMPLE
# ------------------------------------------------------------
def visualize_sample(public_id, split, image_path, mask_path, method_used, compression_ratio=None):
    img_nii, img_data = load_nifti(image_path)
    mask_nii, mask_data = load_nifti(mask_path)

    # güvenlik
    if img_data.shape != mask_data.shape:
        print(f"[WARN] Shape mismatch for {public_id}: image={img_data.shape}, mask={mask_data.shape}")
        return

    z_show = choose_display_slice(mask_data)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # 1) original
    axes[0].imshow(img_data[:, :, z_show].T, cmap="gray", origin="lower")
    axes[0].set_title(f"{public_id} | {split} | Original")
    axes[0].axis("off")

    # 2) overlay
    axes[1].imshow(img_data[:, :, z_show].T, cmap="gray", origin="lower")
    axes[1].imshow(mask_data[:, :, z_show].T, cmap="Reds", alpha=0.35, origin="lower")
    title_2 = f"{public_id} | {method_used} | Overlay"
    if pd.notna(compression_ratio):
        title_2 += f"\ncompression={compression_ratio:.4f}"
    axes[1].set_title(title_2)
    axes[1].axis("off")

    # 3) mask only
    axes[2].imshow(mask_data[:, :, z_show].T, cmap="gray", origin="lower")
    axes[2].set_title(f"{public_id} | Mask Only | z={z_show}")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# HELPER: VISUALIZE RANDOM SAMPLES BY METHOD
# ------------------------------------------------------------
def visualize_random_by_method(df, method_name, n=5, seed=42):
    subset = df[df["method_used"] == method_name].copy()

    if len(subset) == 0:
        print(f"No samples found for method: {method_name}")
        return

    n = min(n, len(subset))
    subset = subset.sample(n=n, random_state=seed)

    print(f"\nShowing {n} random samples for method = {method_name}")
    print(subset[["public_id", "split", "method_used", "compression_ratio"]])

    for _, row in subset.iterrows():
        visualize_sample(
            public_id=row["public_id"],
            split=row["split"],
            image_path=row["image_path_check"],
            mask_path=row["mask_path_check"],
            method_used=row["method_used"],
            compression_ratio=row["compression_ratio"]
        )

# ------------------------------------------------------------
# HELPER: VISUALIZE SPECIFIC public_id
# ------------------------------------------------------------
def visualize_by_public_id(df, public_id):
    row = df[df["public_id"] == public_id]

    if len(row) == 0:
        print(f"Sample not found: {public_id}")
        return

    row = row.iloc[0]

    print(row[[
        "public_id", "split", "method_used",
        "bbox_voxels", "refined_voxels",
        "compression_ratio", "status"
    ]])

    visualize_sample(
        public_id=row["public_id"],
        split=row["split"],
        image_path=row["image_path_check"],
        mask_path=row["mask_path_check"],
        method_used=row["method_used"],
        compression_ratio=row["compression_ratio"]
    )

# ------------------------------------------------------------
# 1) RANDOM REGION_GROWING SAMPLES
# ------------------------------------------------------------
visualize_random_by_method(df_vis, method_name="region_growing", n=5, seed=42)

# ------------------------------------------------------------
# 2) RANDOM FALLBACK SAMPLES
# ------------------------------------------------------------
visualize_random_by_method(df_vis, method_name="fallback_inner_box", n=5, seed=42)

# ------------------------------------------------------------
# 3) OPTIONAL: SINGLE SAMPLE CHECK
# örnek:
# visualize_by_public_id(df_vis, "mela_0007")
# ------------------------------------------------------------

Output hidden; open in https://colab.research.google.com to view.